# 🧠 NLP, Embeddings, Sentiment Analysis · Speech-to-Text · Text-to-Speech
### CEC424 — Introduction to Deep Learning | University of Buea · COT
**Instructor:** TCHAPGA TCHITO C. | **Stack:** TensorFlow/Keras · SpeechRecognition · Whisper · gTTS

---

## Notebook Outline
| # | Section | Key Libraries |
|---|---------|---------------|
| 1 | NLP Fundamentals & Text Preprocessing | Python stdlib, sklearn |
| 2 | Text Representations: BoW · TF-IDF · Embeddings | sklearn, TensorFlow/Keras |
| 3 | Sentiment Analysis — IMDB Dataset | TensorFlow/Keras |
| 4 | Speech-to-Text (STT) | SpeechRecognition, OpenAI Whisper |
| 5 | Text-to-Speech (TTS) | gTTS, pyttsx3 |
| 6 | Full Pipeline: Audio → STT → Sentiment → TTS | All above |

## 📦 Install Dependencies
Run once — restart kernel afterwards if needed.

In [ ]:
import subprocess, sys

packages = [
    "scikit-learn",          # BoW, TF-IDF
    "SpeechRecognition",     # STT via Google Web Speech API
    "openai-whisper",        # Offline STT (Whisper)
    "gTTS",                  # Google Text-to-Speech
    "pyttsx3",               # Offline TTS
    "soundfile",             # Read/write WAV files
    "librosa",               # Audio loading & analysis
    "requests",              # Download sample audio
    "numpy",
    "matplotlib",
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", pkg],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗"
    print(f"{status}  {pkg}")

print("\nAll dependencies checked. Restart kernel if freshly installed.")

## 📥 Core Imports

In [ ]:
import os, re, string, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# NLP / ML
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb

# Audio / Speech
import soundfile as sf
import librosa
import librosa.display

# Display helpers
from IPython.display import Audio, display, HTML

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Keras      : {keras.__version__}")
print("✅  All imports OK")

---
# Part 1 — NLP Fundamentals & Text Preprocessing

**Natural Language Processing (NLP)** is the field of AI that enables machines to understand and generate human language.

### Core preprocessing pipeline
```
Raw text → Lowercase → Remove noise → Tokenize → Remove stopwords → Stem/Lemmatize
```

| Step | Purpose | Example |
|------|---------|---------|
| Lowercasing | Normalize case | `"Hello"` → `"hello"` |
| Punctuation removal | Reduce vocabulary | `"great!"` → `"great"` |
| Tokenization | Split to words | `"I love AI"` → `["I","love","AI"]` |
| Stop-word removal | Remove noise words | remove `"the"`, `"is"`, `"a"` |
| Stemming | Reduce to root | `"running"` → `"run"` |

In [ ]:
# ── Text Preprocessing Pipeline ──────────────────────────────────────────────
STOP_WORDS = {
    "a","an","the","is","it","in","on","at","to","of","and","or","but",
    "not","be","was","were","are","this","that","with","for","as","by",
    "have","has","had","do","did","does","will","would","could","should",
    "may","might","from","into","than","then","so","if","when","where",
    "which","who","whom","there","their","they","them","his","her","its",
    "we","you","i","he","she","my","your","our","been","being"
}

def preprocess(text: str, stem: bool = True) -> str:
    """Full NLP preprocessing pipeline."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # 3. Remove URLs
    text = re.sub(r"https?://\S+", "", text)
    # 4. Keep only letters
    text = re.sub(r"[^a-z\s]", "", text)
    # 5. Tokenize
    tokens = text.split()
    # 6. Remove stopwords & short tokens
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    # 7. Naive stemming (suffix stripping)
    if stem:
        suffixes = ("ing", "tion", "ness", "ment", "ful", "less",
                    "able", "ible", "ed", "er", "est", "ly")
        def stem_word(w):
            for sfx in suffixes:
                if w.endswith(sfx) and len(w) - len(sfx) >= 3:
                    return w[: -len(sfx)]
            return w
        tokens = [stem_word(t) for t in tokens]
    return " ".join(tokens)

# --- Demo ---
samples = [
    "The movie was absolutely BRILLIANT! Best film I've seen this year.",
    "Terrible movie. The acting was wooden and the plot made no sense at all.",
    "An average film — some good moments, but overall disappointing.",
]

print(f"{'Original':<55}  {'Preprocessed'}")
print("-" * 95)
for s in samples:
    print(f"{s:<55}  {preprocess(s)}")

---
# Part 2 — Text Representations & Embeddings

Three approaches to turn text into numbers a model can use:

| Method | Idea | Weakness |
|--------|------|----------|
| **Bag of Words (BoW)** | Count word occurrences | Ignores word order & meaning |
| **TF-IDF** | Weight rare words higher | Still no semantic understanding |
| **Word Embeddings** | Dense vectors capturing meaning | Requires training data |

> **Key insight:** `"king" - "man" + "woman" ≈ "queen"` — embeddings encode semantics!

In [ ]:
# ── Bag of Words vs TF-IDF ────────────────────────────────────────────────────
corpus = [
    "deep learning is powerful for natural language processing",
    "neural networks learn from large amounts of data",
    "convolutional networks are great for image recognition",
    "recurrent networks handle sequential data like speech and text",
    "transformers revolutionized natural language processing and vision",
]

# --- Bag of Words ---
bow_vec  = CountVectorizer(max_features=12)
bow_mat  = bow_vec.fit_transform(corpus).toarray()

# --- TF-IDF ---
tfidf_vec = TfidfVectorizer(max_features=12)
tfidf_mat = tfidf_vec.fit_transform(corpus).toarray()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, mat, vec, title in zip(
    axes,
    [bow_mat, tfidf_mat],
    [bow_vec, tfidf_vec],
    ["Bag of Words (raw counts)", "TF-IDF (weighted)"]
):
    im = ax.imshow(mat, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(vec.get_feature_names_out())))
    ax.set_xticklabels(vec.get_feature_names_out(), rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(corpus)))
    ax.set_yticklabels([f"Doc {i+1}" for i in range(len(corpus))], fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax)

plt.suptitle("Text Representation Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nBoW vocabulary (12 terms):", list(bow_vec.vocabulary_.keys())[:8], "...")

### Word Embeddings with Keras Tokenizer + Embedding Layer

The `Embedding` layer learns a **dense vector** for each word during training.
- Input: integer word index (e.g., `42`)
- Output: dense vector of size `embed_dim` (e.g., `[0.23, -0.81, 0.14, ...]`)

```
Sentence → Tokenize → Pad → [Embedding layer] → Dense vectors per word
```

In [ ]:
# ── Keras Tokenizer & Embedding Layer Demo ───────────────────────────────────
sentences = [
    "deep learning is amazing for NLP tasks",
    "recurrent networks process sequences of text",
    "speech recognition converts audio to words",
    "sentiment analysis classifies positive or negative text",
    "word embeddings capture semantic meaning beautifully",
]

VOCAB_SIZE = 500
MAX_LEN    = 10
EMBED_DIM  = 8

# Step 1: Tokenize
tok = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tok.fit_on_texts(sentences)

# Step 2: Text → sequences of integers
seqs = tok.texts_to_sequences(sentences)

# Step 3: Pad to equal length
padded = pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

print("Word index (first 10):")
print(dict(list(tok.word_index.items())[:10]))
print(f"\nPadded sequences shape: {padded.shape}")
print("\nExample — 'deep learning is amazing for NLP tasks'")
print("  Sequence :", seqs[0])
print("  Padded   :", padded[0])

# Step 4: Tiny Embedding model
embed_model = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    layers.GlobalAveragePooling1D(),
    layers.Dense(4, activation="relu"),
])
embed_model.summary()

# Visualize embedding vectors for first sentence
emb_layer  = embed_model.layers[0]
emb_weights = emb_layer.get_weights()[0]  # shape: (VOCAB_SIZE, EMBED_DIM)

# Plot embeddings of a few key words
words_to_plot = ["deep","learning","speech","text","network","word","audio","nlp"]
indices = [tok.word_index.get(w, 0) for w in words_to_plot]
vecs    = np.array([emb_weights[i] for i in indices])

from sklearn.decomposition import PCA
pca  = PCA(n_components=2)
pts  = pca.fit_transform(vecs)

plt.figure(figsize=(7, 5))
plt.scatter(pts[:, 0], pts[:, 1], c="steelblue", s=120, zorder=3)
for i, word in enumerate(words_to_plot):
    plt.annotate(word, (pts[i, 0]+0.01, pts[i, 1]+0.01), fontsize=11)
plt.title("Word Embeddings — PCA 2D projection (random init)", fontweight="bold")
plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Note: These are random (untrained) embeddings — they become meaningful after training.")

---
# Part 3 — Sentiment Analysis with IMDB Dataset

**Task:** Binary classification — is a movie review *Positive* (1) or *Negative* (0)?

**Dataset:** IMDB — 50,000 reviews (25k train / 25k test), balanced classes, built into Keras.

### Models we'll train
| # | Architecture | Key layer |
|---|-------------|-----------|
| A | Dense (MLP) | GlobalAveragePooling1D |
| B | Bidirectional LSTM | Captures order in both directions |

In [ ]:
# ── Load & Explore IMDB Dataset ───────────────────────────────────────────────
NUM_WORDS = 10_000
MAX_LEN   = 256

print("Loading IMDB dataset …")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS)
print(f"Train: {x_train.shape}  Test: {x_test.shape}")
print(f"Labels — Train: {np.bincount(y_train)}  Test: {np.bincount(y_test)}")

# Decode a sample review
word_index = imdb.get_word_index()
idx2word   = {v+3: k for k, v in word_index.items()}
idx2word.update({0:"<PAD>", 1:"<START>", 2:"<UNK>", 3:"<UNUSED>"})

def decode_review(seq):
    return " ".join(idx2word.get(i, "?") for i in seq)

print(f"\nSample review (label={'POSITIVE' if y_train[0]==1 else 'NEGATIVE'}):")
print(decode_review(x_train[0])[:300], "…")

# Review length distribution
lens = [len(r) for r in x_train]
plt.figure(figsize=(8, 3))
plt.hist(lens, bins=50, color="steelblue", edgecolor="white")
plt.axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN={MAX_LEN}")
plt.xlabel("Review length (tokens)"); plt.ylabel("Count")
plt.title("IMDB — Review Length Distribution", fontweight="bold")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Mean length: {np.mean(lens):.0f}  |  Median: {np.median(lens):.0f}  |  Max: {max(lens)}")

In [ ]:
# ── Pad sequences ─────────────────────────────────────────────────────────────
x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test_pad  = pad_sequences(x_test,  maxlen=MAX_LEN, padding="post", truncating="post")

print(f"x_train_pad shape : {x_train_pad.shape}")
print(f"x_test_pad  shape : {x_test_pad.shape}")
print(f"\nFirst padded review (first 30 tokens): {x_train_pad[0][:30]}")

### Model A — Dense (MLP) with Embedding + GlobalAveragePooling

In [ ]:
# ── Model A: Embedding + GlobalAveragePooling + Dense ────────────────────────
tf.random.set_seed(42)

model_a = keras.Sequential([
    layers.Embedding(NUM_WORDS, 64, input_length=MAX_LEN),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),
], name="Model_A_Dense")

model_a.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
model_a.summary()

cb = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, verbose=0)
hist_a = model_a.fit(
    x_train_pad, y_train,
    validation_split=0.15,
    epochs=10,
    batch_size=256,
    callbacks=[cb],
    verbose=1,
)

loss_a, acc_a = model_a.evaluate(x_test_pad, y_test, verbose=0)
print(f"\n✅  Model A — Test accuracy: {acc_a*100:.2f}%  |  Loss: {loss_a:.4f}")

### Model B — Bidirectional LSTM

In [ ]:
# ── Model B: Bidirectional LSTM ───────────────────────────────────────────────
tf.random.set_seed(42)

model_b = keras.Sequential([
    layers.Embedding(NUM_WORDS, 64, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),
], name="Model_B_BiLSTM")

model_b.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_b.summary()

cb2 = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, verbose=0)
hist_b = model_b.fit(
    x_train_pad, y_train,
    validation_split=0.15,
    epochs=8,
    batch_size=256,
    callbacks=[cb2],
    verbose=1,
)

loss_b, acc_b = model_b.evaluate(x_test_pad, y_test, verbose=0)
print(f"\n✅  Model B — Test accuracy: {acc_b*100:.2f}%  |  Loss: {loss_b:.4f}")

In [ ]:
# ── Training Curves Comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, hist, name, color in zip(
    axes,
    [hist_a, hist_b],
    ["Model A — Dense", "Model B — BiLSTM"],
    ["steelblue", "darkorange"]
):
    epochs_range = range(1, len(hist.history["accuracy"]) + 1)
    ax.plot(epochs_range, hist.history["accuracy"],     color=color, lw=2, label="Train acc")
    ax.plot(epochs_range, hist.history["val_accuracy"], color=color, lw=2, linestyle="--", label="Val acc")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.set_ylim(0.6, 1.0); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Training History — Sentiment Analysis on IMDB", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

# Summary bar
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Dense MLP", "BiLSTM"], [acc_a*100, acc_b*100], color=["steelblue","darkorange"], edgecolor="white")
for i, v in enumerate([acc_a*100, acc_b*100]):
    ax.text(i, v+0.3, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylim(80, 100); ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Model Comparison — Test Accuracy", fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── Sentiment Inference on Custom Text ───────────────────────────────────────
# We use the IMDB word index for inference on new text

def predict_sentiment(text: str, model, word_index, top_k=10_000, maxlen=256) -> dict:
    """Predict sentiment for a raw review string."""
    # Preprocess
    text_clean = re.sub(r"[^a-z\s]", "", text.lower()).split()
    # Encode (shift by 3 as per IMDB convention)
    seq = [min(word_index.get(w, 2)+3, top_k-1) for w in text_clean]
    padded = pad_sequences([seq], maxlen=maxlen, padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0)[0][0])
    label = "POSITIVE 😊" if prob >= 0.5 else "NEGATIVE 😞"
    return {"text": text[:80]+"…", "probability": prob, "label": label}

test_reviews = [
    "This film was an absolute masterpiece! The acting and story were incredible.",
    "Terrible movie. Poor acting, dull plot and a complete waste of two hours.",
    "It was okay, not great but not terrible either. Some nice moments though.",
    "One of the best films I have seen in years. Highly recommend to everyone!",
    "Boring, predictable and utterly disappointing. I nearly fell asleep watching.",
]

print(f"{'Review':<65}  {'Prob':>6}  Label")
print("-" * 95)
for rev in test_reviews:
    result = predict_sentiment(rev, model_b, word_index)
    bar = "█" * int(result["probability"] * 20)
    print(f"{rev[:65]:<65}  {result['probability']:.3f}  {result['label']}")

---
# Part 4 — Speech-to-Text (STT)

**Automatic Speech Recognition (ASR)** converts spoken audio into written text.

### Two approaches covered here

| Approach | Library | Internet? | Quality |
|----------|---------|-----------|---------|
| **Google Web Speech API** | `SpeechRecognition` | ✅ Required | Good |
| **OpenAI Whisper** | `openai-whisper` | ❌ Offline | Excellent |

### How Whisper works
```
Audio waveform → Mel spectrogram → CNN encoder → Transformer decoder → Transcript
```
Whisper was trained on **680,000 hours** of multilingual audio — supports 99 languages including French.

In [ ]:
# ── Download & Visualise a Sample Audio File ─────────────────────────────────
import requests, os

# LibriSpeech sample (public domain, English speech, ~5 sec WAV)
AUDIO_URL  = "https://www.signalogic.com/melp/EngSamples/Orig/male.wav"
AUDIO_PATH = "/tmp/sample_speech.wav"

if not os.path.exists(AUDIO_PATH):
    print("Downloading sample audio …")
    r = requests.get(AUDIO_URL, timeout=15)
    if r.status_code == 200:
        with open(AUDIO_PATH, "wb") as f:
            f.write(r.content)
        print(f"✅  Saved to {AUDIO_PATH}")
    else:
        # Fallback: generate a simple sine-wave "beep" as placeholder
        print("⚠️  Download failed — generating synthetic audio as fallback.")
        import numpy as np
        import soundfile as sf
        sr  = 16000
        t   = np.linspace(0, 3, sr * 3)
        sig = (0.4 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
        sf.write(AUDIO_PATH, sig, sr)
        print(f"✅  Synthetic audio saved to {AUDIO_PATH}")
else:
    print(f"✅  Audio already exists: {AUDIO_PATH}")

# Load and visualise
y_audio, sr = librosa.load(AUDIO_PATH, sr=None)
duration = len(y_audio) / sr
print(f"\nSample rate : {sr} Hz  |  Duration : {duration:.2f}s  |  Samples : {len(y_audio):,}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

# Waveform
axes[0].plot(np.linspace(0, duration, len(y_audio)), y_audio, color="steelblue", lw=0.6)
axes[0].set_title("Audio Waveform", fontweight="bold")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Amplitude")

# Mel Spectrogram
mel = librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=80)
mel_db = librosa.power_to_db(mel, ref=np.max)
img = librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[1], cmap="inferno")
axes[1].set_title("Mel Spectrogram (input to Whisper)", fontweight="bold")
plt.colorbar(img, ax=axes[1], format="%+2.0f dB")
plt.tight_layout(); plt.show()

# Play in notebook
display(Audio(y_audio, rate=sr))

### 4A — SpeechRecognition Library (Google Web Speech API)

In [ ]:
# ── STT: SpeechRecognition (Google Web Speech API) ───────────────────────────
import speech_recognition as sr_lib

recognizer = sr_lib.Recognizer()

# Adjust energy threshold for ambient noise
recognizer.dynamic_energy_threshold = True

print("SpeechRecognition version:", sr_lib.__version__)
print("\nSupported recognition engines:")
engines = ["recognize_google","recognize_sphinx","recognize_whisper","recognize_vosk"]
for e in engines:
    avail = hasattr(recognizer, e)
    print(f"  {'✓' if avail else '✗'}  {e}")

# ── Transcribe from file (requires internet) ─────────────────────────────────
def transcribe_google(audio_path: str, language: str = "en-US") -> str:
    """Transcribe a WAV/AIFF/FLAC file using Google Web Speech API."""
    rec = sr_lib.Recognizer()
    with sr_lib.AudioFile(audio_path) as source:
        rec.adjust_for_ambient_noise(source, duration=0.3)
        audio_data = rec.record(source)
    try:
        text = rec.recognize_google(audio_data, language=language)
        return text
    except sr_lib.UnknownValueError:
        return "[Could not understand audio]"
    except sr_lib.RequestError as e:
        return f"[API error: {e}]"

# Transcribe the sample
print(f"\nTranscribing: {AUDIO_PATH}")
transcript_google = transcribe_google(AUDIO_PATH)
print(f"Transcript (Google): \"{transcript_google}\"")

# ── Microphone demo (real-time) ───────────────────────────────────────────────
print("""
── BONUS: Real-time microphone transcription ─────────────────
Run this block separately after installing pyaudio:
  pip install pyaudio

  r = sr_lib.Recognizer()
  with sr_lib.Microphone() as source:
      print("Speak now…")
      audio = r.listen(source, timeout=5)
  text = r.recognize_google(audio)
  print("You said:", text)
──────────────────────────────────────────────────────────────
""")

### 4B — OpenAI Whisper (Offline, No API Key Needed)

Whisper is a state-of-the-art open-source ASR model. It runs **entirely locally** — no internet or API key required.

| Model size | Parameters | VRAM | Speed |
|-----------|-----------|------|-------|
| `tiny` | 39M | ~1 GB | ⚡⚡⚡ |
| `base` | 74M | ~1 GB | ⚡⚡ |
| `small` | 244M | ~2 GB | ⚡ |
| `medium` | 769M | ~5 GB | 🐢 |
| `large` | 1550M | ~10 GB | 🐢🐢 |

> **For this notebook we use `base`** — fast and accurate enough for demos.

In [ ]:
# ── STT: OpenAI Whisper (fully offline) ──────────────────────────────────────
import whisper
import time

print("Loading Whisper 'base' model (downloads once ~145 MB) …")
t0 = time.time()
whisper_model = whisper.load_model("base")
print(f"✅  Whisper loaded in {time.time()-t0:.1f}s")
print(f"    Encoder layers : {len(whisper_model.encoder.blocks)}")
print(f"    Decoder layers : {len(whisper_model.decoder.blocks)}")
print(f"    Embed dim      : {whisper_model.dims.n_audio_state}")

def transcribe_whisper(audio_path: str, language: str = None) -> dict:
    """Transcribe audio with Whisper. language=None → auto-detect."""
    t0 = time.time()
    opts = {"fp16": False}
    if language:
        opts["language"] = language
    result = whisper_model.transcribe(audio_path, **opts)
    elapsed = time.time() - t0
    return {
        "text":     result["text"].strip(),
        "language": result.get("language", "unknown"),
        "segments": len(result.get("segments", [])),
        "time_s":   round(elapsed, 2),
    }

print(f"\nTranscribing: {AUDIO_PATH}")
result_whisper = transcribe_whisper(AUDIO_PATH)
print(f"\n📝  Whisper transcript : \"{result_whisper['text']}\"")
print(f"    Detected language  : {result_whisper['language']}")
print(f"    Segments           : {result_whisper['segments']}")
print(f"    Inference time     : {result_whisper['time_s']}s")

# Compare results
print("\n── Comparison ──────────────────────────────────────────────")
print(f"Google Web Speech : {transcript_google}")
print(f"Whisper (offline) : {result_whisper['text']}")
print("────────────────────────────────────────────────────────────")

### Whisper Architecture — Detailed Walkthrough

In [ ]:
# ── Visualise Whisper's Mel Spectrogram Processing ───────────────────────────
import whisper

# Load audio the Whisper way (resamples to 16kHz, pads/trims to 30s)
audio_w = whisper.load_audio(AUDIO_PATH)          # float32 array at 16kHz
audio_w = whisper.pad_or_trim(audio_w)            # exactly 30s = 480,000 samples

# Compute log-Mel spectrogram exactly as Whisper does
mel_spec = whisper.log_mel_spectrogram(audio_w)   # shape: (80, 3000)
print(f"Whisper Mel spectrogram shape: {mel_spec.shape}")
print(f"  80 Mel frequency bins  ×  3000 time frames (30s at 10ms hop)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw waveform
axes[0].plot(np.linspace(0, 30, len(audio_w)), audio_w, lw=0.4, color="steelblue")
axes[0].set_title("Raw Waveform (16 kHz, 30s)", fontweight="bold")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Amplitude")

# Log-Mel spectrogram
im = axes[1].imshow(
    mel_spec.numpy(), aspect="auto", origin="lower",
    cmap="inferno", extent=[0, 30, 0, 80]
)
axes[1].set_title("Log-Mel Spectrogram (Whisper input)", fontweight="bold")
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Mel bin")
plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

# Show Whisper architecture layers
print("\n── Whisper 'base' Architecture ─────────────────────────────")
print(f"  Input      : log-Mel spectrogram  (80 × 3000)")
print(f"  Encoder    : 2× CNN stem → 6× Transformer blocks (512-dim)")
print(f"  Decoder    : 6× Transformer blocks → Token logits → text")
print(f"  Vocabulary : 51,864 BPE tokens (multilingual)")
print(f"  Output     : token IDs → decoded text")
print("─────────────────────────────────────────────────────────────")

---
# Part 5 — Text-to-Speech (TTS)

**TTS** converts written text into spoken audio.

| Library | Method | Speed | Quality | Offline |
|---------|--------|-------|---------|---------|
| **gTTS** | Google Translate TTS | Fast | Natural | ❌ |
| **pyttsx3** | OS-level TTS engine | Fast | Robotic | ✅ |

### How neural TTS works
```
Text → G2P (Grapheme-to-Phoneme) → Acoustic model → Waveform synthesiser → Audio
```
Modern systems (like Google/Amazon Polly) use **sequence-to-sequence transformers** for very natural-sounding speech.

In [ ]:
# ── TTS: gTTS (Google Text-to-Speech) ────────────────────────────────────────
from gtts import gTTS
import os

def speak_gtts(text: str, lang: str = "en", output_path: str = "/tmp/tts_out.mp3") -> str:
    """Convert text to speech using gTTS. Returns path to audio file."""
    tts = gTTS(text=text, lang=lang, slow=False)
    tts.save(output_path)
    return output_path

# English
texts_en = [
    "Welcome to the CEC424 Deep Learning course at the University of Buea.",
    "Deep learning is a subset of machine learning that uses neural networks.",
    "Speech recognition converts spoken words into written text automatically.",
]

print("── gTTS English examples ────────────────────────────────────")
for i, text in enumerate(texts_en):
    path = f"/tmp/tts_en_{i}.mp3"
    speak_gtts(text, lang="en", output_path=path)
    print(f"\n[EN {i+1}] \"{text}\"")
    y_tts, sr_tts = librosa.load(path, sr=None)
    display(Audio(y_tts, rate=sr_tts))

# French (gTTS supports many languages!)
texts_fr = [
    "Bienvenue au cours d'apprentissage profond à l'Université de Buea.",
    "La reconnaissance vocale convertit les mots parlés en texte écrit.",
]

print("\n── gTTS French examples ─────────────────────────────────────")
for i, text in enumerate(texts_fr):
    path = f"/tmp/tts_fr_{i}.mp3"
    speak_gtts(text, lang="fr", output_path=path)
    print(f"\n[FR {i+1}] \"{text}\"")
    y_tts, sr_tts = librosa.load(path, sr=None)
    display(Audio(y_tts, rate=sr_tts))

### 5B — pyttsx3 (Offline, System TTS)

`pyttsx3` uses the OS's built-in TTS engine — **no internet needed**.
- macOS: uses `NSSpeechSynthesizer` (AVSpeech)
- Windows: uses `SAPI5`
- Linux: uses `eSpeak`

In [ ]:
# ── TTS: pyttsx3 (Offline) ────────────────────────────────────────────────────
import pyttsx3

engine = pyttsx3.init()

# List available voices
voices = engine.getProperty("voices")
print(f"Available voices: {len(voices)}")
for v in voices[:5]:
    print(f"  ID: {v.id.split('.')[-1]:<20}  Name: {v.name}")

# Configure voice properties
engine.setProperty("rate",   150)    # words per minute (default ~200)
engine.setProperty("volume", 1.0)    # 0.0–1.0

# Save speech to a WAV file (no audio device needed in notebooks)
OUTPUT_WAV = "/tmp/pyttsx3_output.wav"

texts_tts = [
    "Hello! I am an offline text-to-speech engine.",
    "Deep learning uses multiple layers of neural networks.",
    "This speech is generated entirely without an internet connection.",
]

# Speak to file
engine.save_to_file(
    " ".join(texts_tts),
    OUTPUT_WAV
)
engine.runAndWait()
print(f"\n✅  Audio saved to {OUTPUT_WAV}")

# Load and play in notebook
try:
    y_px, sr_px = librosa.load(OUTPUT_WAV, sr=None)
    print(f"Duration: {len(y_px)/sr_px:.2f}s  |  Sample rate: {sr_px} Hz")
    display(Audio(y_px, rate=sr_px))
except Exception as e:
    print(f"Note: {e} — the file may need manual playback in your environment.")

---
# Part 6 — Full Pipeline: Audio → STT → Sentiment → TTS

```
┌──────────────┐    Whisper     ┌─────────────┐   BiLSTM    ┌──────────────────┐    gTTS    ┌──────────────┐
│  Audio input │ ──────────────▶│  Transcript │ ──────────▶│ Sentiment label  │ ──────────▶│ Spoken reply │
└──────────────┘                └─────────────┘             └──────────────────┘            └──────────────┘
```

This pipeline could be used in applications like:
- **Voice-based product review analysis**
- **Customer service call sentiment monitoring**
- **Accessibility tools** (read sentiment of voice messages aloud)

In [ ]:
# ── Generate sample speech audio clips for the pipeline ──────────────────────
# We create TTS audio samples, then feed them back through STT + Sentiment

test_reviews_pipeline = [
    ("This film was absolutely fantastic. One of the best I have ever seen!",  "positive"),
    ("Terrible movie. Complete waste of time. The story made no sense at all.", "negative"),
    ("It was an okay film, decent acting but nothing special or memorable.",    "neutral"),
]

print("Generating speech clips via gTTS …\n")
audio_clips = []
for i, (text, expected) in enumerate(test_reviews_pipeline):
    path = f"/tmp/pipeline_input_{i}.mp3"
    speak_gtts(text, lang="en", output_path=path)
    audio_clips.append((path, text, expected))
    print(f"[{i+1}] {text[:60]}…")
    print(f"      → saved: {path}")

print(f"\n✅  {len(audio_clips)} audio clips ready.")

In [ ]:
# ── Full Pipeline: Audio → Whisper STT → BiLSTM Sentiment → gTTS Reply ───────

def full_pipeline(audio_path: str, stt_model, sentiment_model, wi) -> dict:
    """End-to-end pipeline: audio file → spoken sentiment reply."""
    # Step 1: Speech-to-Text
    stt_result = transcribe_whisper(audio_path)
    transcript = stt_result["text"]

    # Step 2: Sentiment Analysis
    sentiment  = predict_sentiment(transcript, sentiment_model, wi)
    label      = "positive" if sentiment["probability"] >= 0.5 else "negative"
    prob       = sentiment["probability"]

    # Step 3: Generate spoken reply
    if label == "positive":
        reply = f"Sentiment detected: positive, with {prob*100:.0f} percent confidence. Great review!"
    else:
        reply = f"Sentiment detected: negative, with {(1-prob)*100:.0f} percent confidence. The review sounds critical."

    reply_path = audio_path.replace(".mp3", "_reply.mp3").replace(".wav", "_reply.mp3")
    speak_gtts(reply, lang="en", output_path=reply_path)

    return {
        "transcript": transcript,
        "probability": prob,
        "label": label,
        "reply": reply,
        "reply_audio": reply_path,
    }

# ── Run pipeline on each clip ─────────────────────────────────────────────────
print("=" * 70)
print("  FULL PIPELINE: Speech → Text → Sentiment → Spoken Reply")
print("=" * 70)

for i, (path, original_text, expected) in enumerate(audio_clips):
    print(f"\n{'─'*60}")
    print(f"  Clip {i+1} | Expected: {expected.upper()}")
    print(f"  Original text: \"{original_text[:55]}…\"")

    y_in, sr_in = librosa.load(path, sr=None)
    print(f"\n  ► Input audio:")
    display(Audio(y_in, rate=sr_in))

    result = full_pipeline(path, whisper_model, model_b, word_index)
    bar_pos = "█" * int(result["probability"] * 20)
    bar_neg = "░" * (20 - int(result["probability"] * 20))

    print(f"\n  📝 Whisper transcript : \"{result['transcript']}\"")
    print(f"  🎭 Sentiment          : {result['label'].upper()}  ({result['probability']:.3f})")
    print(f"     Positive [{bar_pos}{bar_neg}] Negative")
    print(f"  🔊 Spoken reply       : \"{result['reply']}\"")
    y_rep, sr_rep = librosa.load(result["reply_audio"], sr=None)
    display(Audio(y_rep, rate=sr_rep))

print(f"\n{'='*70}")

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))

# Left: Model accuracy bars
ax1 = fig.add_subplot(1, 3, 1)
ax1.barh(["Dense MLP", "BiLSTM"], [acc_a*100, acc_b*100],
          color=["steelblue","darkorange"], height=0.5)
for i, v in enumerate([acc_a*100, acc_b*100]):
    ax1.text(v+0.2, i, f"{v:.1f}%", va="center", fontweight="bold")
ax1.set_xlim(80, 100); ax1.set_xlabel("Test Accuracy (%)")
ax1.set_title("Sentiment Models\n(IMDB)", fontweight="bold")
ax1.grid(axis="x", alpha=0.3)

# Middle: Pipeline diagram (text)
ax2 = fig.add_subplot(1, 3, 2)
ax2.axis("off")
steps = [
    ("🎙️  Audio Input",     "WAV/MP3 file"),
    ("📝  Whisper STT",      "offline ASR"),
    ("🎭  BiLSTM Sentiment", "pos / neg label"),
    ("🔊  gTTS Reply",       "spoken response"),
]
for i, (step, desc) in enumerate(steps):
    y_pos = 0.85 - i * 0.22
    ax2.text(0.5, y_pos,    step, ha="center", fontsize=11, fontweight="bold",
             transform=ax2.transAxes)
    ax2.text(0.5, y_pos-0.08, desc, ha="center", fontsize=9, color="gray",
             transform=ax2.transAxes)
    if i < len(steps) - 1:
        ax2.text(0.5, y_pos-0.12, "▼", ha="center", fontsize=14, color="steelblue",
                 transform=ax2.transAxes)
ax2.set_title("Full Pipeline", fontweight="bold")

# Right: Technology summary
ax3 = fig.add_subplot(1, 3, 3)
ax3.axis("off")
summary = [
    ("NLP Preprocessing", "Pure Python"),
    ("BoW / TF-IDF",       "scikit-learn"),
    ("Embeddings",         "Keras Embedding"),
    ("Sentiment (MLP)",    f"{acc_a*100:.1f}% acc"),
    ("Sentiment (BiLSTM)", f"{acc_b*100:.1f}% acc"),
    ("STT (online)",       "SpeechRecognition"),
    ("STT (offline)",      "Whisper base"),
    ("TTS (online)",       "gTTS"),
    ("TTS (offline)",      "pyttsx3"),
]
for i, (k, v) in enumerate(summary):
    ax3.text(0.05, 0.95 - i*0.1, f"• {k}:", fontsize=9, fontweight="bold",
             transform=ax3.transAxes, va="top")
    ax3.text(0.55, 0.95 - i*0.1, v, fontsize=9, color="steelblue",
             transform=ax3.transAxes, va="top")
ax3.set_title("Technologies Used", fontweight="bold")

plt.suptitle("CEC424 — NLP + Speech Deep Learning Notebook Summary",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Summary

| Section | Concept | Tool/Library |
|---------|---------|-------------|
| **1** | Text preprocessing pipeline | Pure Python |
| **2** | BoW, TF-IDF, Word Embeddings | sklearn, Keras |
| **3** | Sentiment analysis — IMDB | TensorFlow/Keras BiLSTM |
| **4A** | Speech-to-Text (online) | SpeechRecognition |
| **4B** | Speech-to-Text (offline) | OpenAI Whisper |
| **5A** | Text-to-Speech (online) | gTTS |
| **5B** | Text-to-Speech (offline) | pyttsx3 |
| **6** | Full end-to-end pipeline | All of the above |

### Key Takeaways
- **Embeddings** outperform BoW/TF-IDF because they encode semantic meaning
- **BiLSTM** captures word order (both directions) — better for sequential text
- **Whisper** is production-grade offline ASR supporting 99 languages — ideal for Cameroon's context (French, English, Pidgin)
- **gTTS** produces natural-sounding voice with minimal code
- The full pipeline shows how real-world AI assistants chain these components together

### Next Steps
- Try `whisper.load_model("small")` for better accuracy on noisy audio
- Fine-tune the BiLSTM on domain-specific data (e.g., product reviews in French)
- Explore `Coqui TTS` for higher-quality offline neural TTS
- Add a Flask/FastAPI wrapper to turn the pipeline into a web service

---
*CEC424 — Introduction to Deep Learning | University of Buea · COT | 2025/2026*